# 🫁 PneumoVision — Training Notebook

This notebook trains three CNN models (ResNet-50, EfficientNet-B0, VGG-16) for **3-class pneumonia detection** from chest X-rays.

**Classes:** Normal, Bacterial Pneumonia, Viral Pneumonia

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

---

## Prerequisites
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Have `kaggle.json` API key ready (from Kaggle → Account → Create New API Token)

## Cell 1 — Install Dependencies

In [ ]:
!pip install torch torchvision pytorch-grad-cam scikit-learn matplotlib seaborn reportlab kaggle --quiet
print("All dependencies installed.")

## Cell 2 — Kaggle Dataset Download

Upload your `kaggle.json` file when prompted, then this cell will download and extract the chest X-ray dataset.

In [ ]:
import os
from google.colab import files

# Upload kaggle.json
print("Please upload your kaggle.json file:")
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)

# Move kaggle.json to correct location
import shutil
if 'kaggle.json' in uploaded:
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Download dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia --unzip -p /content/data/
print("\nDataset downloaded successfully.")
print("\nDirectory structure:")
!find /content/data/chest_xray -type d

## Cell 3 — Dataset Inspection & Class Remapping

The original dataset has 2 classes (NORMAL, PNEUMONIA). We remap PNEUMONIA into BACTERIAL and VIRAL based on filename prefixes (`bacteria_*` and `virus_*`).

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import shutil

DATA_ROOT = Path("/content/data/chest_xray")
NEW_DATA_ROOT = Path("/content/data/chest_xray_3class")

CLASS_MAP = {
    "NORMAL": "NORMAL",
    "bacteria": "BACTERIAL",
    "virus": "VIRAL"
}


def remap_dataset(src_root: Path, dst_root: Path):
    """Remap 2-class dataset to 3-class structure."""
    splits = ["train", "val", "test"]
    for split in splits:
        print(f"\n--- {split.upper()} ---")
        for cls in ["NORMAL", "BACTERIAL", "VIRAL"]:
            (dst_root / split / cls).mkdir(parents=True, exist_ok=True)

        # Copy NORMAL
        normal_src = src_root / split / "NORMAL"
        if normal_src.exists():
            for f in normal_src.iterdir():
                if f.is_file():
                    shutil.copy(f, dst_root / split / "NORMAL" / f.name)

        # Copy and split PNEUMONIA
        pneumonia_src = src_root / split / "PNEUMONIA"
        if pneumonia_src.exists():
            for f in pneumonia_src.iterdir():
                if not f.is_file():
                    continue
                fname = f.name.lower()
                if "bacteria" in fname:
                    shutil.copy(f, dst_root / split / "BACTERIAL" / f.name)
                elif "virus" in fname:
                    shutil.copy(f, dst_root / split / "VIRAL" / f.name)
                else:
                    # Fallback: assign to BACTERIAL if prefix is ambiguous
                    shutil.copy(f, dst_root / split / "BACTERIAL" / f.name)

        # Print counts
        for cls in ["NORMAL", "BACTERIAL", "VIRAL"]:
            count = len(list((dst_root / split / cls).iterdir()))
            print(f"  {split}/{cls}: {count} images")


print("Remapping dataset to 3-class structure...")
remap_dataset(DATA_ROOT, NEW_DATA_ROOT)
print("\nDone. New dataset root:", NEW_DATA_ROOT)

# Visualize sample images from each class
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, cls in enumerate(["NORMAL", "BACTERIAL", "VIRAL"]):
    sample_files = list((NEW_DATA_ROOT / "train" / cls).iterdir())[:1]
    if sample_files:
        img = Image.open(sample_files[0]).convert("RGB")
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(cls, fontsize=14, fontweight='bold')
        axes[i].axis("off")
plt.suptitle("Sample X-Rays by Class", fontsize=16)
plt.tight_layout()
plt.savefig("/content/sample_images.png", dpi=150)
plt.show()

## Cell 4 — Data Transforms, Datasets, DataLoaders with WeightedRandomSampler

Handles class imbalance via `WeightedRandomSampler` and weighted `CrossEntropyLoss`.

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from collections import Counter

BATCH_SIZE = 32
IMG_SIZE = 224
DATA_DIR = NEW_DATA_ROOT

# Define transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATA_DIR / "val",   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(DATA_DIR / "test",  transform=val_test_transforms)

CLASS_NAMES = train_dataset.classes  # ['BACTERIAL', 'NORMAL', 'VIRAL']
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", CLASS_NAMES)

# Compute class weights for WeightedRandomSampler (handles imbalance)
targets = [s[1] for s in train_dataset.samples]
class_counts = Counter(targets)
class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Compute loss weights for CrossEntropyLoss
total = sum(class_counts.values())
loss_weights = torch.tensor([total / (NUM_CLASSES * class_counts[i]) for i in range(NUM_CLASSES)], dtype=torch.float32)
print("Loss weights:", loss_weights)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## Cell 5 — Model Definitions (ResNet50, EfficientNetB0, VGG16)

Three CNN architectures with frozen backbones and custom classification heads.

In [ ]:
import torch.nn as nn
from torchvision import models


def build_resnet50(num_classes: int) -> nn.Module:
    """ResNet50 with fine-tunable final layer."""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    # Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False
    # Replace final FC layer
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes)
    )
    return model


def build_efficientnetb0(num_classes: int) -> nn.Module:
    """EfficientNetB0 with fine-tunable classifier."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_vgg16(num_classes: int) -> nn.Module:
    """VGG16 with fine-tunable classifier."""
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, num_classes)
    )
    return model


def unfreeze_all(model: nn.Module):
    """Unfreeze all parameters for full fine-tuning phase."""
    for param in model.parameters():
        param.requires_grad = True


MODEL_BUILDERS = {
    "resnet50":       build_resnet50,
    "efficientnetb0": build_efficientnetb0,
    "vgg16":          build_vgg16,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Cell 6 — Training Loop with Early Stopping

Two-phase training:
1. **Epochs 1–10:** Only train the classification head (backbone frozen)
2. **Epochs 11+:** Unfreeze all layers, lower learning rate for full fine-tuning

Includes early stopping with patience=5 on validation loss.

In [ ]:
import time
import copy
from torch.optim.lr_scheduler import CosineAnnealingLR


def train_model(
    model_name: str,
    model: nn.Module,
    train_loader,
    val_loader,
    num_epochs: int = 25,
    unfreeze_epoch: int = 10,
    patience: int = 5,
    device=DEVICE
):
    """
    Full training loop with two-phase learning and early stopping.

    Returns:
        model: Best model (loaded with best weights)
        history: dict with train_loss, val_loss, train_acc, val_acc lists
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=loss_weights.to(device))

    # Phase 1 optimizer: only trainable params (head)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4, weight_decay=1e-5
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        # Unfreeze at unfreeze_epoch
        if epoch == unfreeze_epoch + 1:
            unfreeze_all(model)
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-5)
            scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs - unfreeze_epoch)
            print(f"  >> Epoch {epoch}: Unfroze all layers. LR reduced to 1e-5.")

        # --- Training Phase ---
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc  = running_correct / total

        # --- Validation Phase ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc   = val_correct / val_total
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch:02d}/{num_epochs}] | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  >> Early stopping at epoch {epoch}.")
                break

    elapsed = time.time() - start_time
    print(f"  >> Training complete in {elapsed/60:.1f} minutes.")

    model.load_state_dict(best_weights)
    return model, history


print("Training function defined.")

## Cell 7 — Train All Three Models

Sequentially trains ResNet-50, EfficientNet-B0, and VGG-16. Saves weights to `/content/models/`.

In [ ]:
import os

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/metrics", exist_ok=True)

all_histories = {}

for model_name, builder in MODEL_BUILDERS.items():
    print(f"\n{'='*60}")
    print(f"Training: {model_name.upper()}")
    print('='*60)

    model = builder(NUM_CLASSES)
    trained_model, history = train_model(
        model_name=model_name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=25,
        unfreeze_epoch=10,
        patience=5
    )

    # Save weights
    weight_path = f"/content/models/{model_name}_pneumo.pth"
    torch.save({
        "model_state_dict": trained_model.state_dict(),
        "class_names": CLASS_NAMES,
        "num_classes": NUM_CLASSES,
        "model_name": model_name,
    }, weight_path)
    print(f"  >> Weights saved: {weight_path}")

    all_histories[model_name] = history

print("\n" + "="*60)
print("All models trained and saved.")
print("="*60)

## Cell 8 — Evaluation: Metrics, Confusion Matrix, ROC Curves

Evaluates each model on the test set. Generates:
- Classification report
- Confusion matrix (PNG)
- ROC curves with AUC scores (PNG)
- Training/validation loss & accuracy curves (PNG)
- Metrics JSON files

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve, classification_report
)
from sklearn.preprocessing import label_binarize


def evaluate_model(model, loader, device=DEVICE):
    """Run inference on entire loader, return predictions and probabilities."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def save_confusion_matrix(y_true, y_pred, class_names, model_name, out_dir):
    """Save confusion matrix as PNG."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix — {model_name.upper()}", fontsize=14, fontweight='bold')
    plt.ylabel("Actual", fontsize=12)
    plt.xlabel("Predicted", fontsize=12)
    plt.tight_layout()
    path = f"{out_dir}/{model_name}_confusion_matrix.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  Confusion matrix saved: {path}")
    return cm


def save_roc_curves(y_true, y_probs, class_names, model_name, out_dir):
    """Save one-vs-rest ROC curves for all classes."""
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    plt.figure(figsize=(8, 6))
    auc_scores = {}

    colors = ['#00C49A', '#FF4B4B', '#FFB347']
    for i, (cls, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        auc = roc_auc_score(y_bin[:, i], y_probs[:, i])
        auc_scores[cls] = round(auc, 4)
        plt.plot(fpr, tpr, color=color, lw=2, label=f"{cls} (AUC = {auc:.3f})")

    plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title(f"ROC Curves (One-vs-Rest) — {model_name.upper()}", fontsize=14, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    path = f"{out_dir}/{model_name}_roc_curves.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  ROC curves saved: {path}")
    return auc_scores


def save_loss_curves(history, model_name, out_dir):
    """Save training/validation loss and accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history["train_loss"]) + 1)
    axes[0].plot(epochs, history["train_loss"], 'b-o', label='Train Loss', markersize=3)
    axes[0].plot(epochs, history["val_loss"],   'r-o', label='Val Loss',   markersize=3)
    axes[0].set_title(f"Loss Curves — {model_name.upper()}", fontweight='bold')
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], 'b-o', label='Train Acc', markersize=3)
    axes[1].plot(epochs, history["val_acc"],   'r-o', label='Val Acc',   markersize=3)
    axes[1].set_title(f"Accuracy Curves — {model_name.upper()}", fontweight='bold')
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    path = f"{out_dir}/{model_name}_loss_curves.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print(f"  Loss curves saved: {path}")


# ---- Run Evaluation for All Models ----
all_metrics = {}

for model_name, builder in MODEL_BUILDERS.items():
    print(f"\n{'='*50}\nEvaluating: {model_name.upper()}\n{'='*50}")

    # Reload model
    checkpoint = torch.load(f"/content/models/{model_name}_pneumo.pth", map_location=DEVICE)
    model = builder(NUM_CLASSES).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    y_true, y_pred, y_probs = evaluate_model(model, test_loader)

    # Compute metrics
    metrics = {
        "accuracy":          round(accuracy_score(y_true, y_pred), 4),
        "precision_macro":   round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "recall_macro":      round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "f1_macro":          round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "f1_weighted":       round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "class_names":       CLASS_NAMES,
    }

    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # Plots
    save_confusion_matrix(y_true, y_pred, CLASS_NAMES, model_name, "/content/metrics")
    auc_scores = save_roc_curves(y_true, y_probs, CLASS_NAMES, model_name, "/content/metrics")
    save_loss_curves(all_histories[model_name], model_name, "/content/metrics")

    metrics["auc_scores"] = auc_scores
    all_metrics[model_name] = metrics

    # Save metrics JSON
    with open(f"/content/metrics/{model_name}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"  Metrics JSON saved.")

# Save combined metrics
with open("/content/metrics/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print("\nAll metrics saved to /content/metrics/")

## Cell 9 — Package and Download

Creates ZIP archives of model weights and metrics for download. Place the extracted files into:
- `streamlit_app/models/` ← `.pth` files
- `streamlit_app/metrics/` ← JSON + PNG files

In [ ]:
import shutil

# Zip everything for download
shutil.make_archive("/content/pneumovision_export", "zip", "/content", "models")
shutil.make_archive("/content/pneumovision_metrics", "zip", "/content", "metrics")

print("Export packages ready:")
print("  /content/pneumovision_export.zip  — model weights (.pth files)")
print("  /content/pneumovision_metrics.zip — metrics JSONs + PNGs")
print("\nDownload both files and place them in:")
print("  streamlit_app/models/   <- .pth files")
print("  streamlit_app/metrics/  <- JSON + PNG files")

# Download
from google.colab import files
files.download("/content/pneumovision_export.zip")
files.download("/content/pneumovision_metrics.zip")

print("\n✅ Download initiated. Check your browser downloads.")